# Zen Chan Graph Ideas

In [ ]:
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.graph_objects as go

## 1. Visit Duration vs. Time of Day (Violin Plot)

In [ ]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT time_of_day, visit_duration_sec FROM visits', conn)
conn.close()

fig = px.violin(df, x='time_of_day', y='visit_duration_sec', box=True, points='all', 
                title='Visit Duration vs. Time of Day', 
                labels={'time_of_day': 'Time of Day', 'visit_duration_sec': 'Visit Duration (seconds)'})
fig.show()

## 2. Pre-label Co-occurrence (Heatmap)

In [ ]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT visit_id, pre_labels FROM visits', conn)
conn.close()

# Create a co-occurrence matrix
co_occurrence = pd.crosstab(df['visit_id'], df['pre_labels'])
co_occurrence = co_occurrence.T.dot(co_occurrence)
fig = px.imshow(co_occurrence, title='Pre-label Co-occurrence')
fig.show()

## 3. Domain Diversity Over Time (Line Chart)

In [ ]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT visit_datetime, domain FROM visits', conn)
conn.close()

df['visit_date'] = pd.to_datetime(df['visit_datetime']).dt.date
domain_diversity = df.groupby('visit_date')['domain'].nunique().reset_index()

fig = px.line(domain_diversity, x='visit_date', y='domain', title='Domain Diversity Over Time',
              labels={'visit_date': 'Date', 'domain': 'Number of Unique Domains'})
fig.show()

## 4. Mood Transitions (Sankey Diagram)

In [ ]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT visit_datetime, mood FROM visits ORDER BY visit_datetime', conn)
mood_colors_df = pd.read_sql_query('SELECT mood, color_hex FROM moods', conn)
conn.close()

mood_colors = mood_colors_df.set_index('mood')['color_hex'].to_dict()

df['mood_from'] = df['mood']
df['mood_to'] = df['mood'].shift(-1)
mood_transitions = df.groupby(['mood_from', 'mood_to']).size().reset_index(name='value')

# Thresholding
threshold = 5
mood_transitions = mood_transitions[mood_transitions['value'] >= threshold]

# Two-column layout
all_moods_from = list(pd.unique(mood_transitions['mood_from']))
all_moods_to = list(pd.unique(mood_transitions['mood_to']))

all_moods = all_moods_from + all_moods_to
mood_map = {mood: i for i, mood in enumerate(all_moods)}

fig = go.Figure(data=[go.Sankey(
    node = dict (
      pad = 15,
      thickness = 20,
      line = dict(color = 'black', width = 0.5),
      label = [mood + ' (from)' for mood in all_moods_from] + [mood + ' (to)' for mood in all_moods_to],
      color = [mood_colors.get(mood, 'blue') for mood in all_moods_from] + [mood_colors.get(mood, 'blue') for mood in all_moods_to]
    ),
    link = dict (
      source = mood_transitions['mood_from'].map(mood_map),
      target = mood_transitions['mood_to'].map(lambda x: mood_map.get(x) + len(all_moods_from) if pd.notna(x) else -1),
      value = mood_transitions['value']
  ))])

fig.update_layout(title_text='Mood Transitions (Threshold = 5)', font_size=10)
fig.show()

## 5. Mood Transition Probability (Heatmap)

In [ ]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT visit_datetime, mood FROM visits ORDER BY visit_datetime', conn)
conn.close()

df['mood_from'] = df['mood']
df['mood_to'] = df['mood'].shift(-1)
transition_matrix = pd.crosstab(df['mood_from'], df['mood_to'], normalize='index')

fig = px.imshow(transition_matrix, text_auto=True, title='Mood Transition Probability')
fig.show()